In [ ]:
from statsmodels.stats.proportion import proportions_ztest
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_parquet('ready_all_langs.parquet')

In [ ]:
agg = (
    df
    .assign(success=lambda d: d["hard_toxic_neg"].eq(1).astype(int))
    .groupby(["language", "platform"])
    .agg(k=("success", "sum"),
         n=("success", "size"))
    .reset_index()
)


In [ ]:
def ztest_on_aggregates(tbl: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    rows = []

    overall = (
        tbl.groupby("platform")[["k", "n"]]
        .sum()
        .reindex(["twitch", "youtube"])
    )

    if not overall["k"].isna().any() and not overall["n"].isna().any():
        k_glob = overall["k"].to_numpy()
        n_glob = overall["n"].to_numpy()

        z_glob, p_glob = proportions_ztest(k_glob, n_glob)
        prop_t, prop_y = k_glob / n_glob * 100
        diff_glob      = prop_t - prop_y
        h_glob = 2 * (np.arcsin(np.sqrt(k_glob[0] / n_glob[0])) -
                      np.arcsin(np.sqrt(k_glob[1] / n_glob[1])))

        rows.append(dict(
            language      = "overall",
            n_twitch      = n_glob[0], k_twitch = k_glob[0],
            n_youtube     = n_glob[1], k_youtube = k_glob[1],
            prop_twitch   = round(prop_t, 2),
            prop_youtube  = round(prop_y, 2),
            diff_pct      = round(diff_glob, 2),
            z             = z_glob,
            p_value       = p_glob,
            significant   = p_glob < alpha,
            cohens_h      = round(h_glob, 3)
        ))

    for lang, sub in tbl.groupby("language"):
        sub = sub.set_index("platform").reindex(["twitch", "youtube"])

        if sub["k"].isna().any() or sub["n"].isna().any():
            continue

        k = sub["k"].to_numpy()
        n = sub["n"].to_numpy()

        z, p = proportions_ztest(k, n)

        prop_t, prop_y = k / n * 100
        diff = prop_t - prop_y
        h = 2 * (np.arcsin(np.sqrt(k[0] / n[0])) -
                 np.arcsin(np.sqrt(k[1] / n[1])))

        rows.append(dict(
            language      = lang,
            n_twitch      = n[0], k_twitch = k[0],
            n_youtube     = n[1], k_youtube = k[1],
            prop_twitch   = round(prop_t, 2),
            prop_youtube  = round(prop_y, 2),
            diff_pct      = round(diff, 2),
            z             = z,
            p_value       = p,
            significant   = p < alpha,
            cohens_h      = round(h, 3)
        ))

    df_res = pd.DataFrame(rows)
    lang_order = (["overall"] +
                  df_res.loc[df_res["language"] != "overall"]
                        .sort_values("diff_pct", ascending=False)["language"].tolist())
    df_res["language"] = pd.Categorical(df_res["language"], categories=lang_order, ordered=True)
    return df_res.sort_values("language").reset_index(drop=True)


In [5]:
result = ztest_on_aggregates(agg, alpha=0.05)
result

,language,n_twitch,k_twitch,n_youtube,k_youtube,prop_twitch,prop_youtube,diff_pct,z,p_value,significant,cohens_h
0,overall,2859222,91946,1414201,83738,3.22,5.92,-2.71,-132.546806,0.000000e+00,True,-0.131
1,Russian,220947,11179,110472,4766,5.06,4.31,0.75,9.452271,3.315569e-21,True,0.035
2,Thai,15117,194,7546,92,1.28,1.22,0.06,0.407622,6.835515e-01,False,0.006
3,Chinese,43026,727,14325,238,1.69,1.66,0.03,0.227654,8.199155e-01,False,0.002
4,Arabic,14536,138,7262,68,0.95,0.94,0.01,0.093401,9.255850e-01,False,0.001
5,Turkish,10175,31,5068,37,0.30,0.73,-0.43,-3.712806,2.049744e-04,True,-0.061
6,Italian,33433,1259,16707,732,3.77,4.38,-0.62,-3.327655,8.758015e-04,True,-0.031
7,Japanese,121811,1448,60834,1299,1.19,2.14,-0.95,-15.665375,2.608872e-55,True,-0.075
8,German,132568,2217,66242,2042,1.67,3.08,-1.41,-20.471146,3.893526e-93,True,-0.094
9,Portuguese,147395,3269,73696,2698,2.22,3.66,-1.44,-19.739383,9.897889e-87,True,-0.086


In [ ]:
df.head()

In [ ]:
import pandas as pd
import numpy as np
from math import sqrt
from typing import List, Dict, Tuple
from scipy import stats

def _cohens_d(x: np.ndarray, y: np.ndarray) -> float:
    nx, ny = len(x), len(y)
    pooled_sd = sqrt(((nx - 1) * np.var(x, ddof=1)
                     + (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2))
    return (np.mean(x) - np.mean(y)) / pooled_sd


def welch_t_ci(
    x: np.ndarray,
    y: np.ndarray,
    n_boot: int = 5_000,
    alpha: float = 0.05,
) -> Dict[str, float | Tuple[float, float] | int]:
    t_stat, p_val = stats.ttest_ind(x, y, equal_var=False)
    d_obs = _cohens_d(x, y)

    rng = np.random.default_rng(seed=42)
    boots = [_cohens_d(
                rng.choice(x, size=x.size, replace=True),
                rng.choice(y, size=y.size, replace=True)
             )
             for _ in range(n_boot)]
    ci_low, ci_high = np.percentile(
        boots, [100 * alpha / 2, 100 * (1 - alpha / 2)]
    )

    return {
        "t": t_stat, "p": p_val, "d": d_obs,
        "ci_low": ci_low, "ci_high": ci_high,
        "n1": len(x), "n2": len(y),
    }

def compare_hard_toxic_neg(
    twitch_df: pd.DataFrame,
    youtube_df: pd.DataFrame,
    n_boot: int = 5_000,
    alpha: float = 0.05,
) -> pd.DataFrame:
    x = twitch_df["hard_toxic_neg"].astype(float).to_numpy()
    y = youtube_df["hard_toxic_neg"].astype(float).to_numpy()
    res = welch_t_ci(x, y, n_boot=n_boot, alpha=alpha)
    return pd.DataFrame([{ "test": "hard_toxic_neg", **res }])


def compare_hard_toxic_neg_languages(
    twitch_df: pd.DataFrame,
    youtube_df: pd.DataFrame,
    n_boot: int = 5_000,
    alpha: float = 0.05,
) -> pd.DataFrame:
    common_langs = sorted(
        set(twitch_df["language"]).intersection(youtube_df["language"])
    )

    rows: List[Dict[str, object]] = []
    for lang in common_langs:
        t_sub = twitch_df[twitch_df["language"] == lang]
        y_sub = youtube_df[youtube_df["language"] == lang]
        if t_sub.empty or y_sub.empty:
            continue
        stats_row = welch_t_ci(
            t_sub["hard_toxic_neg"].astype(float).to_numpy(),
            y_sub["hard_toxic_neg"].astype(float).to_numpy(),
            n_boot=n_boot, alpha=alpha,
        )
        rows.append({ "language": lang, **stats_row })

    return pd.DataFrame(rows)

twitch_df  = df[df["platform"] == "twitch"]
youtube_df = df[df["platform"] == "youtube"]

print("=== Global comparison ===")
print(compare_hard_toxic_neg(twitch_df, youtube_df).to_string(index=False))

print("\n=== By-language comparison ===")
print(compare_hard_toxic_neg_languages(twitch_df, youtube_df).to_string(index=False))


=== Global comparison ===
          test           t   p         d    ci_low   ci_high      n1      n2
hard_toxic_neg -120.659109 0.0 -0.136544 -0.138672 -0.134376 2859222 1414201

=== By-language comparison ===
  language           t             p         d    ci_low   ci_high      n1     n2
    Arabic    0.093609  9.254214e-01  0.001342 -0.027033  0.028241   14536   7262
   Chinese    0.228593  8.191871e-01  0.002196 -0.017168  0.020499   43026  14325
   English -115.675759  0.000000e+00 -0.170935 -0.173716 -0.168210 1708556 853761
    French  -40.084246  0.000000e+00 -0.212701 -0.222650 -0.202626  138382  69073
    German  -18.597300  4.512327e-77 -0.097506 -0.107394 -0.087286  132568  66242
   Italian   -3.248537  1.161222e-03 -0.031531 -0.050219 -0.012765   33433  16707
  Japanese  -14.271282  3.684761e-46 -0.077825 -0.088274 -0.067623  121811  60834
    Polish   -8.294350  1.167779e-16 -0.104461 -0.129780 -0.080438   22385  11153
Portuguese  -18.243847  2.918037e-74 -0.089133 -0.

# Graph

In [ ]:
import plotly.graph_objects as go

result["language"] = result["language"].replace("overall", "Overall")
result["significant"] = result["p_value"] < 0.05
result_sorted = result.sort_values("cohens_h", key=abs, ascending=False)

formatted_labels = [
    f"<b>{lang}</b>" if lang == "Overall" else lang
    for lang in result_sorted["language"]
]

bar_colors = [
    "#d62728" if sig else "#7f7f7f"
    for sig in result_sorted["significant"]
]

bar_trace = go.Bar(
    x=result_sorted["cohens_h"],
    y=formatted_labels,
    orientation="h",
    marker_color=bar_colors,
    hovertemplate="Cohen's h: %{x:.3f}<extra></extra>",
    showlegend=False
)

legend_traces = [
    go.Bar(
        x=[None],
        y=[None],
        marker_color="#d62728",
        name="p < 0.05 (Significant)"
    ),
    go.Bar(
        x=[None],
        y=[None],
        marker_color="#7f7f7f",
        name="p ≥ 0.05 (Not significant)"
    )
]

fig = go.Figure(data=[bar_trace] + legend_traces)

fig.update_layout(
    title="Effect Size (Cohen's h) by Language",
    title_x=0.5,
    xaxis_title="Cohen's h",
    yaxis_title="Language",
    xaxis=dict(
        tickmode='linear',
        tick0=-0.2,
        dtick=0.05,
        range=[-0.22, 0.08],
        zeroline=True,
        zerolinewidth=2,
        zerolinecolor='gray',
        showgrid=True,
        gridcolor='lightgrey'
    ),
    yaxis=dict(showgrid=False),
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=600,
    width=950,
    margin=dict(l=100, r=200), 
    legend=dict(
        x=1.02,
        y=1,
        traceorder="normal",
        bgcolor="white",
        bordercolor="black",
        borderwidth=0.5
    )
)

fig.show()

In [ ]:

result["language"] = result["language"].replace("overall", "Overall")

row_colors = []
for lang, sig in zip(result["language"], result["significant"]):
    if lang == "Overall":
        row_colors.append('#d7ecff')  
    elif sig:
        row_colors.append('#f3e8fd') 
    else:
        row_colors.append('white')    

import plotly.graph_objects as go

fig = go.Figure(
    data=[
        go.Table(
            header=dict(
                values=[
                    "<b>Language</b>", "<b>Twitch %</b>", "<b>YouTube %</b>",
                    "<b>Diff %</b>", "<b>p-value</b>", "<b>Cohen's h</b>", "<b>Significant</b>"
                ],
                fill_color='whitesmoke',
                align='center',
                font=dict(size=12, color='black'),
                line_color='lightgrey',
                height=32
            ),
            cells=dict(
                values=[
                    result["language"],
                    result["prop_twitch"],
                    result["prop_youtube"],
                    result["diff_pct"],
                    [f"{p:.2e}" for p in result["p_value"]],
                    result["cohens_h"],
                    result["significant"]
                ],
                fill_color=[row_colors] * 7,
                align='center',
                font=dict(size=11, color='black'),
                line_color='lightgrey',
                height=28
            )
        )
    ]
)

fig.update_layout(
    title="Statistical Summary Table by Language",
    title_x=0.5,
    height=700,
    width=900,
    paper_bgcolor="white"
)

fig.show()